# Expansion Trajectories & Influence Spheres

Visualize how civilizations expand across the galaxy with dynamic expansion lines and influence zones.

## New Features:
- **Expansion Trajectories**: Lines showing parent→colony connections
- **Influence Spheres**: Translucent spheres showing civilization reach
- **Dynamic Growth**: Both features animate over time showing expansion
- **Color-Coded**: Each civilization has a unique color from perceptually uniform palettes

---

## 1. Setup and Run Simulation

We'll run a small optimistic simulation to see expansion in action.

In [ ]:
from great_silence import GalaxySimulation, SimulationConfig, configure_m1_max_threading
from great_silence.notebook import configure_notebook_display

configure_m1_max_threading()
configure_notebook_display()

print("✓ Setup complete")

In [ ]:
# Create optimistic simulation with snapshots
config = SimulationConfig.with_preset('optimistic')
config.galaxy.total_stars = 10_000
config.simulation.duration_gyr = 5.0
config.simulation.save_snapshots = True
config.simulation.snapshot_interval_myr = 500  # Snapshot every 500 Myr

# Run simulation
print(f"Running simulation: {config.galaxy.total_stars} stars, {config.simulation.duration_gyr} Gyr")
sim = GalaxySimulation(config, seed=42)
sim.run(verbose=True)

# Show results
stats = sim.get_statistics()
print(f"\n=== Results ===")
print(f"Total civilizations: {stats['total_civilizations']}")
print(f"Active: {stats['active_civilizations']}, Extinct: {stats['extinct_civilizations']}")
print(f"Total colonies: {sum(len(c.colonized_stars) for c in sim.civilizations)}")
print(f"Snapshots: {len(sim.snapshots)}")

## 2. Static Visualization with Trajectories

Create a 3D visualization showing final expansion state with trajectory lines.

In [ ]:
from great_silence.visualization import Interactive3DVisualizer

viz = Interactive3DVisualizer(sim)

# Create figure with expansion trajectories
fig = viz.create_static_figure(
    subsample_stars=5000,
    show_stars=True,
    show_active=True,
    show_extinct=True,
    show_trajectories=True,  # Show parent→colony lines
    show_spheres=False
)

fig.update_layout(title="Galaxy with Expansion Trajectories (Parent→Colony Lines)")
fig.show()

print("\n💡 Tip: Click and drag to rotate the galaxy!")
print("Lines show how each civilization expanded from its home world to colonies.")

## 3. Static Visualization with Influence Spheres

Show translucent spheres representing each civilization's area of influence.

In [ ]:
# Create figure with influence spheres
fig = viz.create_static_figure(
    subsample_stars=5000,
    show_stars=True,
    show_active=True,
    show_extinct=True,
    show_trajectories=False,
    show_spheres=True  # Show translucent influence zones
)

fig.update_layout(title="Galaxy with Influence Spheres (Civilization Reach)")
fig.show()

print("\n💡 Sphere radius = distance to furthest colony")
print("Translucent spheres show each civilization's area of influence.")

## 4. Combined View: Trajectories + Spheres

Show both features together for maximum insight.

In [ ]:
# Create figure with both features
fig = viz.create_static_figure(
    subsample_stars=5000,
    show_stars=True,
    show_active=True,
    show_extinct=False,  # Hide extinct for clarity
    show_trajectories=True,
    show_spheres=True
)

fig.update_layout(title="Galaxy: Combined Trajectories & Influence Spheres")
fig.show()

print("\n💡 Each civilization has a unique color")
print("Lines show expansion paths, spheres show total reach.")

## 5. Animated Visualization: Watch Expansion Over Time

The real magic: see trajectories and spheres grow dynamically as civilizations colonize!

In [ ]:
# Create animated figure with both features
print("Creating animation... (this may take a moment)")

fig_anim = viz.create_animated_figure(
    subsample_stars=5000,
    show_stars=True,
    show_hazards=True,
    show_trajectories=True,  # Grows as colonies arrive
    show_spheres=True  # Expands as civilizations reach farther
)

fig_anim.update_layout(title="Civilization Expansion Over Time")
fig_anim.show()

print("\n🎬 Controls:")
print("  - Click Play button to watch expansion")
print("  - Drag timeline slider to jump to specific time")
print("  - Trajectories appear as colonies are established")
print("  - Spheres grow as civilizations expand outward")

## 6. Interactive Widget: Full Control

Use the Galaxy3DExplorer widget for complete control over all visualization layers.

In [ ]:
from great_silence.notebook import NotebookSimulationRunner, Galaxy3DExplorer

# Wrap simulation in runner
runner = NotebookSimulationRunner(config, seed=42)
runner.simulation = sim
runner.results = sim.get_statistics()

# Create explorer widget
explorer = Galaxy3DExplorer(runner)
explorer.display()

print("\n💡 Widget Features:")
print("  - Toggle any visualization layer on/off")
print("  - Enable 'Show expansion trajectories' checkbox")
print("  - Enable 'Show influence spheres' checkbox")
print("  - Click 'Create Animation' to see dynamic growth")
print("  - Export interactive plots to HTML")

## 7. Analyze Expansion Patterns

Let's look at which civilizations expanded the most.

In [ ]:
import pandas as pd
import numpy as np

# Build dataframe of expansion statistics
expansion_data = []

for civ in sim.civilizations:
    n_colonies = len(civ.colonized_stars) - 1  # Exclude home world
    
    if n_colonies > 0:
        # Calculate influence radius
        home_pos = sim.galaxy.positions[civ.parent_star_idx]
        max_distance = 0
        
        for colony_idx in civ.colonized_stars:
            if colony_idx != civ.parent_star_idx:
                colony_pos = sim.galaxy.positions[colony_idx]
                distance = np.linalg.norm(colony_pos - home_pos)
                max_distance = max(max_distance, distance)
        
        expansion_data.append({
            'civ_id': civ.civ_id,
            'colonies': n_colonies,
            'max_reach_kpc': max_distance,
            'is_active': civ.is_active,
            'kardashev': civ.kardashev_level
        })

if expansion_data:
    df = pd.DataFrame(expansion_data)
    df = df.sort_values('max_reach_kpc', ascending=False)
    
    print("=== Top 10 Civilizations by Expansion ===\n")
    print(df.head(10).to_string(index=False))
    
    print(f"\n=== Expansion Statistics ===")
    print(f"Total expanding civilizations: {len(df)}")
    print(f"Average colonies per civilization: {df['colonies'].mean():.1f}")
    print(f"Average expansion reach: {df['max_reach_kpc'].mean():.2f} kpc")
    print(f"Maximum expansion reach: {df['max_reach_kpc'].max():.2f} kpc")
else:
    print("No civilizations expanded beyond their home world.")

## 8. Export Interactive Visualization

Save the combined visualization as a standalone HTML file.

In [ ]:
from great_silence.notebook import export_interactive_plot
from pathlib import Path

# Create output directory
Path('output').mkdir(exist_ok=True)

# Create comprehensive figure
fig_export = viz.create_static_figure(
    subsample_stars=10000,
    show_stars=True,
    show_active=True,
    show_extinct=True,
    show_deaths=True,
    show_hazards=True,
    show_trajectories=True,
    show_spheres=True
)

# Export to HTML
html_path = export_interactive_plot(
    fig_export, 
    'output/expansion_visualization.html',
    auto_open=False
)

print(f"✓ Exported to: {html_path}")
print("\nOpen this file in a web browser to:")
print("  - Rotate and zoom the 3D galaxy")
print("  - Toggle layers in the legend")
print("  - Hover over points for details")
print("  - Share with colleagues!")

---

## Summary: What We Learned

### Expansion Trajectories
- Lines connect each civilization's home world to its colonies
- Color-coded by civilization for easy tracking
- Shows expansion paths through the galaxy
- Grows dynamically in animations as colonies arrive

### Influence Spheres
- Translucent spheres centered at home worlds
- Radius = distance to furthest colony
- Shows total area of influence/control
- Expands in animations as civilizations colonize farther

### Use Cases
- **Research**: Understand expansion patterns and collision zones
- **Presentations**: Create compelling visualizations of galactic colonization
- **Education**: Show how civilizations compete for territory
- **Analysis**: Identify which civilizations expanded most successfully

---

## Next Steps

Explore other notebooks:
- **01_quickstart_production_workflow.ipynb**: Run different simulation scenarios
- **02_interactive_exploration.ipynb**: Deep-dive into results
- **03_animation_generation.ipynb**: Create custom animations

Try different parameters:
- Increase galaxy size to see larger-scale patterns
- Use 'optimistic' preset for more expansion
- Adjust snapshot interval for smoother animations
- Compare expansion under different Great Filter scenarios

Happy exploring! 🚀🌌